# Subtitles Showcase

This notebook demonstrates how to extract and process text from `.srt` subtitle files, clean the text, remove duplicates, and finally extract vocabulary tokens (verbs and other words) using the Natural Language API.

In [1]:
%load_ext autoreload

In [4]:
%autoreload 2
from setup_imports import *  # noqa: F401,F403


In [30]:


from src.subtitles import read_subtitles, process_subtitles, get_subtitle_tokens
from src.nlp import get_text_tokens, get_verbs_and_vocab
from src.phrases.phrase_model import get_unique_tokens_from_phrases, Phrase
from src.phrases.generation import generate_phrases_from_vocab_dict
from src.phrases.search import get_phrases_by_collection, find_phrases_by_vocab_dict
from src.utils import save_text_file, load_text_file, save_json, load_json

## 1. Reading Subtitles
We use `pysrt` to read the subtitles from the `data/subtitles/` directory.

In [31]:
subtitle_file = "../data/subtitles/Trouble_sv_only.srt"

try:
    subs = read_subtitles(subtitle_file)
    print(f"Successfully loaded {len(subs)} subtitle entries.")
    print("\nFirst 3 entries:")
    for i in range(min(3, len(subs))):
        print(f"{i+1}: {subs[i].text}")
except Exception as e:
    print(f"Error reading subtitles: {e}")

Successfully loaded 1543 subtitle entries.

First 3 entries:
1: [visslar en melodi]
2: Men snälla, tyst.
3: Du väntar här. När jag messar dig
kommer du in med resten av grejerna.


## 2. Processing Subtitles
We will now clean the subtitles by removing Closed Captioning (CC) info enclosed in square brackets `[like this]`, replace newlines with spaces, strip extra whitespace, and remove duplicates to get a unique list of phrases.

In [32]:
unique_phrases = process_subtitles(subs, language_code="sv", split_on_space=True)

print(f"Total unique phrases extracted: {len(unique_phrases)}")
print("\nSample of 10 unique phrases:")
for phrase in unique_phrases[:100]:
    print(f"{phrase}")

Total unique phrases extracted: 1313

Sample of 10 unique phrases:
Men snälla tyst
Du väntar här När jag messar dig kommer du in med resten av grejerna
Ja Capite
Ska vi sticka
Vad är det här för nåt
Det är ett kilo det här Vi sa 30  Mm
Vi har dem Men pengarna först
Vi står utanför Bamboo Bamboo
Det är bekräftat de misstänkta befinner sig i lokalen
Kör
Enhet A vänster Enhet B bakvägen
Ja Uppfattat
Portkoden
1490
Vänster Vänster Vänster
Ja grabbar det här kan ni In och plocka dem
så tar vi en AW på Stopet sen
Då kallar jag in grejerna
Vapen Vapen
Vad händer Vad fan händer
Nej
Angripare avväpnad Vi avancerar
Polis Polis Visa händerna
Ner på golvet Släpp vapnen Den tog i västen Det är lugnt
Nu tar vi dem
Ner på golvet Ner på golvet
Händerna över huvudet Håll käften
Var fan är alla droger
Det saknas en massa Var fan är resten
Hej
Hej hej ta det lugnt Okej okej
Chilla okej okej Vänta
Här
Här Ta grejerna Skjut inte
Åh shit
Mina damer och herrar detta är kapten Conny Rundkvist från cockpit
Som

## 3. Extracting Tokens
Using `src.nlp`, we'll run a sample of these phrases through the Google Cloud Natural Language API to extract lemmas and group them into `verbs` and `vocab`.

In [33]:
# To avoid long API wait times in this showcase, we'll process just the first 20 phrases.
sample_phrases = unique_phrases[:20]

print(f"Extracting vocabulary from {len(unique_phrases)} phrases...")
vocab_dict = get_verbs_and_vocab(unique_phrases, language="sv-se")

print("\n--- Extracted Verbs ---")
print(vocab_dict.get('verbs', []))

print("\n--- Extracted Vocab (Non-Verbs) ---")
print(vocab_dict.get('vocab', []))

Extracting vocabulary from 1313 phrases...
2026-05-03 10:07:51 - audio-language-trainer - INFO - nlp.py:83 - _load_spacy_model: loaded spaCy model 'sv_core_news_lg' for language 'sv'

--- Extracted Verbs ---
['skola', 'ramla', 'stämma', 'spela', 'planera', 'jävlar', 'störa', 'strömma', 'visa', 'fattar', 'falla', 'måste', 'förvänta', 'snacka', 'hämta', 'förflytta', 'byta', 'glömma', 'hålla', 'svara', 'rör', 'pulla', 'befinna', 'lösa', 'upprepa', 'flytta', 'blöda', 'håll', 'bromsa', 'hända', 'åka', 'lägga', 'behålla', 'träffa', 'grepa', 'störta', 'fundera', 'ringa', 'blicka', 'syssla', 'jävla', 'inackordera', 'agera', 'spänna', 'inträffa', 'ge', 'leta', 'påträffa', 'följa', 'tala', 'försvann', 'fixaa', 'sitta', 'okej', 'raka', 'läsa', 'flög', 'välja', 'saa', 'sköt', 'valsa', 'jobba', 'grabeb', 'kolla', 'släppa', 'förändra', 'luka', 'improvisera', 'hej', 'hoppa', 'flyga', 'vilja', 'so', 'vinna', 'såg', 'fokusera', 'bo', 'köpa', 'garantera', 'vadå', 'ställa', 'be', 'förstår', 'äta', 'äga',

In [34]:
p, m = find_phrases_by_vocab_dict(vocab_dict, language="sv-SE")

In [35]:
from src.subtitles import get_subtitle_tokens
missing_verbs = m['verbs']
missing_valid_verbs = get_subtitle_tokens(missing_verbs, "sv", min_length=3)
missing_vocab = m['vocab']
missing_valid_vocab = get_subtitle_tokens(missing_vocab, "sv", min_length=4)

In [36]:
vocab_dict_save = {"verbs" : missing_valid_verbs, "vocab" : missing_valid_vocab}

In [38]:
from utils import save_json, load_json
#save_json(vocab_dict_save, file_path="../data/trouble_2024_vocab_dict.json")
subtitle_vocab_dict = load_json("../data/trouble_2024_vocab_dict.json")

In [39]:
len(subtitle_vocab_dict["verbs"]), len(subtitle_vocab_dict["vocab"])

(119, 264)

# Generate new foreign phrases

In [24]:
# tags=["film::Trouble_2024"]
subset_dict = {"verbs": subtitle_vocab_dict["verbs"][:10], "vocab": subtitle_vocab_dict["vocab"][:50]}


In [25]:
subtitle_phrases = generate_phrases_from_vocab_dict(subset_dict, language="sv-SE", max_iterations=60)

2026-05-03 09:51:25 - audio-language-trainer - INFO - generation.py:73 - Starting verb phrase generation. 10 verbs to process.
2026-05-03 09:51:25 - audio-language-trainer - INFO - generation.py:120 -   [1/10] Generating phrases for verb: 'ramla'
2026-05-03 09:51:28 - audio-language-trainer - INFO - generation.py:120 -   [2/10] Generating phrases for verb: 'strömma'
2026-05-03 09:51:33 - audio-language-trainer - INFO - generation.py:120 -   [3/10] Generating phrases for verb: 'jävlar'
2026-05-03 09:51:35 - audio-language-trainer - INFO - generation.py:120 -   [4/10] Generating phrases for verb: 'fattar'
2026-05-03 09:51:39 - audio-language-trainer - INFO - generation.py:120 -   [5/10] Generating phrases for verb: 'måste'
2026-05-03 09:51:42 - audio-language-trainer - INFO - generation.py:120 -   [6/10] Generating phrases for verb: 'förstöra'
2026-05-03 09:51:45 - audio-language-trainer - INFO - generation.py:120 -   [7/10] Generating phrases for verb: 'snacka'
2026-05-03 09:51:49 - aud

In [26]:
save_text_file(subtitle_phrases[0], "../data/Trouble_24_subtitle_phrases_10_verbs_50_vocab.txt")

In [28]:
subtitle_phrases[0]

['Jag ramlar ofta',
 'Hon ramlade igår',
 'Kommer du att ramla?',
 'Vattnet strömmar snabbt här',
 'Floden strömmade över bron',
 'Regnet kommer strömma ner',
 'Folk strömmar till konserten',
 'Jag strömmar musik hemma',
 'Jag fattar inte',
 'Fattade du det?',
 'Hon kommer att fatta',
 'Fatta tag i repet',
 'Jag måste gå nu',
 'Vi var tvungna att vänta',
 'Du kommer att måste betala',
 'Du förstör allt',
 'Han förstörde min cykel',
 'Jag kommer att förstöra det',
 'Jag snackar med henne',
 'Vi snackade om filmen',
 'Ska du snacka med chefen?',
 'Jag förflyttar möblerna nu',
 'Vi förflyttade oss snabbt',
 'Ska du förflytta dig?',
 'Hon förflyttar patienten försiktigt',
 'De förflyttar sig norrut',
 'Rör inte maten',
 'Hon rörde vid bordet',
 'Jag kommer röra om',
 'Det rör mig inte',
 'Rör i grytan',
 'Jag pullar min hund',
 'Hon pullade honom i armen',
 'Ska du pulla mig?',
 'Pulla inte i håret',
 'Kerstin i den mysiga garderoben',
 'ungefär fem kontanter',
 'yes från polisen',
 'en im

In [29]:
ALL_PHRASES = []
for foreign in subtitle_phrases[0]:
    p = Phrase.create_from_foreign(foreign, language="sv-SE", split_on_space=True, tags=["film::Trouble_2024"])
    p.upload()
    ALL_PHRASES.append(p)

(y) Google Translate API client initialized
2026-05-03 09:56:03 - audio-language-trainer - INFO - nlp.py:83 - _load_spacy_model: loaded spaCy model 'sv_core_news_lg' for language 'sv'
2026-05-03 09:56:03 - audio-language-trainer - INFO - phrase_model.py:418 - (y) Created translation for sv-SE: Jag ramlar ofta - tokens: ['Jag', 'ramlar', 'ofta']
2026-05-03 09:56:03 - audio-language-trainer - INFO - phrase_model.py:452 - Uploading phrase i_fall_often_e2238e with all translations to Firestore and GCS
2026-05-03 09:56:03 - audio-language-trainer - INFO - phrase_model.py:1112 - Uploading all multimedia for en-GB translation
2026-05-03 09:56:03 - audio-language-trainer - INFO - phrase_model.py:1112 - Uploading all multimedia for sv-SE translation
2026-05-03 09:56:05 - audio-language-trainer - INFO - phrase_model.py:418 - (y) Created translation for sv-SE: Hon ramlade igår - tokens: ['Hon', 'ramlade', 'igår']
2026-05-03 09:56:05 - audio-language-trainer - INFO - phrase_model.py:452 - Uploadin

In [ ]:
lm1000 = get_phrases_by_collection("LM1000")

In [ ]:
lm1000_str = [(x._get_translation('sv-SE').text) for x in lm1000[:10]]

In [13]:
results = find_phrases_by_vocab_dict(
vocab_dict={"verbs": ["springa", "äta"], "vocab": ["hund", "bil"]},
language="sv-SE",
collection="LM1000")
for p in results:
    print(p.english, p.translations["sv-SE"].text)

2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_syntax: language not supported by Google NLP: sv-SE
2026-04-19 15:12:14 - audio-language-trainer - INFO - nlp.py:189 - extract_lemmas_and_pos: falling back to spaCy for language 'sv-SE'
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:197 - extract_lemmas_and_pos: neither Google NLP nor spaCy could process phrase for language 'sv-SE': Ett bättre alternativ
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_syntax: language not supported by Google NLP: sv-SE
2026-04-19 15:12:14 - audio-language-trainer - INFO - nlp.py:189 - extract_lemmas_and_pos: falling back to spaCy for language 'sv-SE'
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:197 - extract_lemmas_and_pos: neither Google NLP nor spaCy could process phrase for language 'sv-SE': En cykel i låg växel
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_synt

{'verbs': [],
 'vocab': [],
 'tokens': ['och',
  'rent',
  'flicka',
  'person',
  'under',
  'för',
  'kort',
  'billig',
  'bättre',
  'färg',
  'pappersark',
  'ljus',
  'förändring',
  'Ett',
  'i',
  'cykel',
  'bänken',
  'genomtänkt',
  'flaska',
  'pojke',
  'varje',
  'glasring',
  'En',
  'växel',
  'svar',
  'alternativ',
  'låg',
  'en',
  'väster']}

In [ ]:
get_

In [ ]:
lm1000_tokens = get_unique_tokens_from_phrases(lm1000, "sv-SE")
lm1000_tokens = list(map(lambda x: x.lower(), lm1000_tokens))

In [ ]:
len(film_tokens)

In [ ]:
len(lm1000_tokens)

In [ ]:
(set(map(lambda x: x[:-1],film_tokens)).difference(set(lm1000_tokens)))

In [ ]:
film_tokens